In [ ]:
import os
import re
import fnmatch
import glob
import sys
module_path = os.path.abspath(os.path.join('./src/python'))
sys.path.append(module_path)
import afim
import xesmf             as xe
import numpy             as np
import pandas            as pd
import xarray            as xr
import cosima_cookbook   as cc
import matplotlib.pyplot as plt
import cartopy.crs       as ccrs
import cmocean
from dask.distributed import Client
from datetime         import datetime,timedelta
import importlib
%matplotlib inline

In [ ]:
importlib.reload(afim)

In [ ]:
cice_prep = afim.cice_prep(os.path.join(os.path.expanduser('~'), 'src', 'python', 'afim_on_gadi.json'))
cice_prep.acom2_coallate_for_cice_forcing()

In [ ]:
xr.open_dataset('/scratch/jk72/da1339/model_forcing/acom2_ocn_frcg_cice6_2005.nc')

# construct yearly OCN dataset from AC-OM2-01
1. request data using cosima cookbook
2. compute sea surface slope using eta 
3. pull in the qdp ... requires compute_ocean_qdp_from 

In [ ]:
session   = cc.database.create_session()
eta       = cc.querying.getvar(expt='01deg_jra55v140_iaf', variable='sea_level',session=session, frequency='1 daily', start_time='2005-01-01 12:00:00',end_time='2005-12-31 12:00:00')
sst       = cc.querying.getvar(expt='01deg_jra55v140_iaf', variable='surface_temp',session=session, frequency='1 daily', start_time='2005-01-01 12:00:00',end_time='2005-12-31 12:00:00')
sss       = cc.querying.getvar(expt='01deg_jra55v140_iaf', variable='surface_salt',session=session, frequency='1 daily', start_time='2005-01-01 12:00:00',end_time='2005-12-31 12:00:00')
mld       = cc.querying.getvar(expt='01deg_jra55v140_iaf', variable='mld',session=session, frequency='1 daily', start_time='2005-01-01 12:00:00',end_time='2005-12-31 12:00:00')
u         = cc.querying.getvar(expt='01deg_jra55v140_iaf', variable='u',session=session, frequency='1 daily', start_time='2005-01-01 12:00:00',end_time='2005-12-31 12:00:00').isel(st_ocean=0)
v         = cc.querying.getvar(expt='01deg_jra55v140_iaf', variable='v',session=session, frequency='1 daily', start_time='2005-01-01 12:00:00',end_time='2005-12-31 12:00:00').isel(st_ocean=0)
G_CICE    = xr.open_dataset(cice_prep.F_G_CICE_original)
dhdx      = afim.compute_ocn_sfc_slope(eta, G_CICE.hte.data, direction='x')
dhdy      = afim.compute_ocn_sfc_slope(eta, G_CICE.htn.data, direction='y')
qdp       = afim.
LN,LT     = np.meshgrid(sst.xt_ocean,sst.yt_ocean)

In [ ]:
data_vars = {'qdp' : (['time','nj','ni'],qdp.qdp.data,{'units':'W/m^2','long_name':'deep ocean heat flux','_FillValue':-2e8}),
             'T'   : (['time','nj','ni'],sst.surface_temp.data,{'units':'K','long_name':'sea surface temperature','_FillValue':-2e8}),
             'S'   : (['time','nj','ni'],sss.surface_salt.data,{'units':'psu','long_name':'sea surface salinity','_FillValue':-2e8}),
             'hblt': (['time','nj','ni'],mld.mld.data,{'units':'m','long_name':'mixed layer depth','_FillValue':-2e8}),
             'U'   : (['time','nj','ni'],uocn.u.data,{'units':'m/s','long_name':'meridional surface current','_FillValue':-2e8}),
             'V'   : (['time','nj','ni'],vocn.v.data,{'units':'m/s','long_name':'zonal surface current','_FillValue':-2e8}),
             'dhdx': (['time','nj','ni'],ssl.dhdx.data,{'units':'m','long_name':'zonal sea surface slope','_FillValue':-2e8}),
             'dhdy': (['time','nj','ni'],ssl.dhdy.data,{'units':'m','long_name':'meridional sea surface slope','_FillValue':-2e8})}
coords = {'xc'   : (['nj','ni'],LN,{'units':'degrees_east'}),
          'yc'   : (['nj','ni'],LT,{'units':'degrees_north'}),
          'time' : (['time'],sst.time.data,{'long_name':'time','cartesian_axis':'T','calendar_type':'GREGORIAN'})}
attrs = {'creation_date': datetime.now().strftime('%Y-%m-%d %H'),
         'conventions'  : "CCSM data model domain description -- for CICE6 standalone 'NCAR' ocean option",
         'title'        : "daily averaged ocean forcing from ACCESS-OM2-01 output for CICE6 standalone ocean forcing",
         'source'       : "ACCESS-OM2-01",
         'comment'      : "source files found on gadi, /g/data/cj50/access-om2/raw-output/access-om2-01/01deg_jra55v140_iaf",
         'calendar'     : "standard",
         'note1'        : "grid is ACCESS-OM2 t-grid",
         'note2'        : "u,v are left un-interpolated to t-grid (i.e. they are from ACCESS-OM2 u-grid)",
         'note3'        : "source files were concatenated using NCO tools, see python afim module method concat_access_om2",
         'note4'        : "dhdx,dhdy are described in ACCESS-OM2 output as effective sea level (eta_t + patm/(rho0*g)) on T cells",
         'note4a'       : "I have computed sea surface slope using python afim module function compute_ocn_sfc_slope, which assumes input is eta_t",
         'author'       : 'Daniel P Atwater',
         'email'        : 'daniel.atwater@utas.edu.au'}
OCN       = xr.Dataset(data_vars=data_vars,coords=coords,attrs=attrs)
enc_dict  = {'shuffle':True,'zlib':True,'complevel':5}
write_job = OCN.to_netcdf(OCN, unlimited_dims=['time'], compute=False,
                          encoding={'qdp':enc_dict, 'T':enc_dict, 'S':enc_dict, 'hblt':enc_dict,
                                    'U':enc_dict, 'V':enc_dict, 'dhdx':enc_dict, 'dhdy':enc_dict})
with ProgressBar():f
    print(f"Writing to {P_ATM_tmp}")
    write_job.compute()

In [ ]:
sst.isel(time=180).plot(figsize=(20,10))
sss.isel(time=180).plot(figsize=(20,10))
mld.isel(time=180).plot(figsize=(20,10))
u.isel(time=180).plot(figsize=(20,10))
v.isel(time=180).plot(figsize=(20,10))
dhdx.isel(time=180).plot(figsize=(20,10))
dhdy.isel(time=180).plot(figsize=(20,10))

In [ ]:
xr.open_dataset('/scratch/jk72/da1339/qdp/tmp/qdp_yd129.nc').qdp.plot(figsize=(20,10))

In [ ]:
variables = cc.querying.get_variables(session, experiment='01deg_jra55v140_iaf')
list(variables.name)

In [ ]:
IC_gx3 = xr.open_dataset('/g/data/jk72/da1339/cice-dirs/input/CICE_data/ic/gx3/iced_gx3_v6.2005-01-01.nc')
G_gx3 = xr.open_dataset('/g/data/jk72/da1339/cice-dirs/input/CICE_data/grid/gx3/grid_gx3.nc')

In [ ]:
IC_gx3['lat'] = (['nj','ni'],G_gx3.ulat.values*(180/np.pi),{'units':'degrees_north','_FillValue':-2e8})
IC_gx3['lon'] = (['nj','ni'],G_gx3.ulon.values*(180/np.pi),{'units':'degrees_east','_FillValue':-2e8})
IC_gx3['htn'] = (['nj','ni'],G_gx3.htn.values*100,{'units':'m','_FillValue':-2e16})
IC_gx3['hte'] = (['nj','ni'],G_gx3.hte.values*100,{'units':'m','_FillValue':-2e16})
IC_gx3['hus'] = (['nj','ni'],G_gx3.hus.values*100,{'units':'m','_FillValue':-2e16})
IC_gx3['angle'] = (['nj','ni'],G_gx3.angle.values*(180/np.pi),{'units':'degrees','_FillValue':-2e16})

In [ ]:
G_CICE        = xr.open_dataset(cice_prep.F_G_CICE_original)
G_CICE['lat'] = (['ny','nx'],G_CICE.tlat.values*(180/np.pi),{'units':'degrees_north','_FillValue':-2e8})
G_CICE['lon'] = (['ny','nx'],G_CICE.tlon.values*(180/np.pi),{'units':'degrees_east','_FillValue':-2e8})
G_CICE        = G_CICE.drop(('ulat','ulon','tlat','tlon','clon_t','clat_t','clat_u','clon_u','angle','uarea'))
G_CICE        = G_CICE.assign_coords(xt_ocean=G_CICE['lon'][0,:],yt_ocean=G_CICE['lat'][:,0])
G_CICE        = G_CICE.set_index(nx='xt_ocean',ny='yt_ocean')

In [ ]:
rg = xe.Regridder(IC_gx3, G_CICE, method='patch')
IC = rg(IC_gx3)

In [ ]:
IC.aicen.isel(ncat=0).plot(figsize=(20,10))

In [ ]:
G_BRAN         = xr.open_dataset("/g/data/gb6/BRAN/BRAN2020/static/ocean_grid.nc")
LN,LT          = np.meshgrid(G_BRAN.xt_ocean.values,G_BRAN.yt_ocean.values)
G_BRANt['lon'] = (['yt_ocean','xt_ocean'],LN,{'units':'degrees_east'})
G_BRANt['lat'] = (['yt_ocean','xt_ocean'],LT,{'units':'degrees_north'})
G_BRANt        = G_BRANt.drop(('xu_ocean','yu_ocean','hu','kmu','umask','tmask','st_edges_ocean','Time','st_ocean'))
G_BRANt

In [ ]:
rg = xe.Regridder(G_BRAN, G_CICE, 'bilinear', filename='/g/data/jk72/da1339/grids/weights/bran_ac_om2_01_bilinear.nc', reuse_weights=False)

In [ ]:
start_date = cice_prep.regrid_start_date
dt_obj = datetime.strptime(start_date,'%Y-%m-%d').date() + pd.DateOffset(years=0)
spdts  = cice_prep.time_series_day_month_start_or_end(start_date=dt_obj.strftime('%Y-%m-%d'),month_start=False)
stdts  = cice_prep.time_series_day_month_start_or_end(start_date=dt_obj.strftime('%Y-%m-%d'))
yr_str = cice_prep.define_year_string(stdts[0])
G_CICE = cice_prep.cice_grid_prep()
G_BRAN = cice_prep.bran_grid_prep()
rg     = xe.Regridder(G_BRAN, G_CICE, method='bilinear',  periodic=True,  filename=cice_prep.F_BRAN_weights, reuse_weights=True)
temp_n = xr.open_mfdataset( os.path.join(cice_prep.D_BRAN,'daily','ocean_temp_{yr:s}*.nc'.format(yr=yr_str)) , chunks={'lat':100,'lon':100} )#, parallel=True, chunks={'time':1})
temp   = rg(temp_n)
temp   = temp.drop(('nv'))
salt_n = xr.open_mfdataset( os.path.join(cice_prep.D_BRAN,'daily','ocean_salt_{yr:s}*.nc'.format(yr=yr_str)) , chunks={'lat':100,'lon':100} )#, parallel=True, chunks={'time':1})
salt   = rg(salt_n)
salt   = salt.drop(('nv'))
mld_n  = xr.open_mfdataset( os.path.join(cice_prep.D_BRAN,'daily','ocean_mld_{yr:s}*.nc'.format(yr=yr_str)) , chunks={'lat':100,'lon':100} )#, parallel=True, chunks={'time':1})
mld    = rg(mld_n)
mld    = mld.drop('nv')
uocn_n = xr.open_mfdataset( os.path.join(cice_prep.D_BRAN,'daily','ocean_u_{yr:s}*.nc'.format(yr=yr_str)) , chunks={'lat':100,'lon':100} )#, parallel=True, chunks={'time':1})
uocn_n = uocn_n.isel(st_ocean=0)
uocn_n = uocn_n.rename(xu_ocean='xt_ocean',yu_ocean='yt_ocean')
uocn   = rg(uocn_n)  
uocn   = uocn.drop(('st_ocean','nv','st_edges_ocean'))
vocn_n = xr.open_mfdataset( os.path.join(cice_prep.D_BRAN,'daily','ocean_v_{yr:s}*.nc'.format(yr=yr_str)) , chunks={'lat':100,'lon':100} )#, parallel=True, chunks={'time':1})
vocn_n = vocn_n.isel(st_ocean=0)
vocn_n = vocn_n.rename(xu_ocean='xt_ocean',yu_ocean='yt_ocean')
vocn   = rg(vocn_n)
vocn   = vocn.drop(('st_ocean','nv','st_edges_ocean'))
eta_n  = xr.open_mfdataset( os.path.join(cice_prep.D_BRAN,'daily','ocean_eta_t_{yr:s}*.nc'.format(yr=yr_str)) , chunks={'lat':100,'lon':100} )#, parallel=True, chunks={'time':1})
eta    = rg(eta_n)
eta    = eta.drop(('nv'))
sst    = temp.sel(st_ocean=0,method='nearest')
sss    = salt.sel(st_ocean=0,method='nearest')
dhdx   = afim.compute_ocn_sfc_slope(eta, G_CICE, direction='x', grid_scale_factor=100)
dhdy   = afim.compute_ocn_sfc_slope(eta, G_CICE, direction='y', grid_scale_factor=100)
# compute qdp
print("computing temperature derivative over time")
dTdt    = temp.differentiate("Time")
print("computing atmospheric surface heat flux")
G_ERA5    = cice_prep.era5_grid_prep()
rg        = xe.Regridder(G_ERA5, G_CICE, method='bilinear', periodic=True, filename=cice_prep.F_ERA5_weights, reuse_weights=True)
dt0       = datetime.strptime(start_date,'%Y-%m-%d') + timedelta(days=(0*365))
lw_nat    = xr.open_mfdataset( os.path.join(cice_prep.D_ERA5,'msdwlwrf',str(dt0.year),'*.nc') , chunks={'lat':100,'lon':100} )
sw_nat    = xr.open_mfdataset( os.path.join(cice_prep.D_ERA5,'msdwswrf',str(dt0.year),'*.nc') , chunks={'lat':100,'lon':100} )
msshf_nat = xr.open_mfdataset( os.path.join(cice_prep.D_ERA5,'msshf',str(dt0.year),'*.nc') , chunks={'lat':100,'lon':100} )
mslhf_nat = xr.open_mfdataset( os.path.join(cice_prep.D_ERA5,'mslhf',str(dt0.year),'*.nc') , chunks={'lat':100,'lon':100} )
F_net_nat = sw_nat.msdwswrf - lw_nat.msdwlwrf - mslhf_nat.mslhf - msshf_nat.msshf
F_net_rg  = rg(F_net_nat)
F_net_rg  = F_net_rg.assign_coords(lon=G_CICE.lon[0,:],lat=G_CICE.lat[:,0])
F_net_rg  = F_net_rg.rename({'ny':'yt_ocean','nx':'xt_ocean'})
F_net_rg  = F_net_rg.resample(time='1D').mean()
print("computing ocean heat capacity at the mixed layer depth")
cp   = afim.compute_ocn_heat_capacity_at_depth(salt.salt, temp.temp, mld.mld, mld.ny)
#cp   = cp.drop(('st_ocean','nv','st_edges_ocean'))
print("computing ocean density at the mixed layer depth")
rho  = afim.compute_ocn_density_at_depth(salt.salt, temp.temp, mld.mld, mld.ny)
#rho  = rho.drop(('st_ocean','nv','st_edges_ocean'))

In [ ]:
rho  = afim.compute_ocn_density_at_depth(salt.salt, temp.temp, mld.mld, mld.ny)

In [ ]:
cp.isel(st_ocean=0,Time=0).plot(figsize=(20,10))

In [ ]:
rho.isel(st_ocean=0,Time=0).plot(figsize=(20,10))